# Experiments Analysis

Retrieve and filter LangSmith evaluation runs for flexible post-hoc analysis.

In [1]:
import sys
import os
import json
from collections import Counter

sys.path.insert(0, os.path.abspath(".."))

from typing import Any, Callable, Dict, List, Optional

from langsmith import Client

client = Client()

In [2]:
def load_experiment_runs(
    experiment_prefix: str = "",
    dataset_name: str = "Dataset v1.",
) -> List[Dict[str, Any]]:
    """
    Retrieve all evaluation runs from LangSmith experiments and return them
    as a flat list of dicts ready for filtering and inspection.

    Each record contains:
        experiment         — experiment (project) name
        task_id            — the eval task identifier
        thread_id          — the pipeline checkpoint thread id
        run_id             — LangSmith run id
        scores             — dict of {score_key: float} from all evaluators
        <score_key>        — every score also flattened to top level for easy filtering
        final_code         — generated code from the pipeline state
        retrieval_context  — RAG context from the pipeline state
        comments           — dict of {feedback_key: comment_string} from evaluators
        statements         — parsed JSON from first available comment (or None if no valid JSON)
        outputs            — raw LangSmith run outputs (full pre_computed_state inside)

    Args:
        experiment_prefix: Only include experiments whose name starts with this string.
                           Empty string means include all experiments for the dataset.
        dataset_name:      LangSmith dataset name used to scope experiments.
    """
    records: List[Dict[str, Any]] = []

    projects = list(client.list_projects(reference_dataset_name=dataset_name))
    if experiment_prefix:
        projects = [p for p in projects if p.name.startswith(experiment_prefix)]

    print(f"Found {len(projects)} experiment(s) matching prefix '{experiment_prefix}'")

    for project in projects:
        runs = list(client.list_runs(project_name=project.name, is_root=True))

        if not runs:
            continue

        # Bulk-fetch all feedback for this project in one call
        run_ids = [str(r.id) for r in runs]
        feedback_by_run: Dict[str, Dict[str, Any]] = {rid: {} for rid in run_ids}
        for fb in client.list_feedback(run_ids=run_ids):
            rid = str(fb.run_id)

            if fb.score is not None:
                feedback_by_run[rid][fb.key] = fb.score
            
            # Store the comment for each feedback key as well
            if fb.comment:
                if f"{fb.key}_comment" not in feedback_by_run[rid]:
                    feedback_by_run[rid][f"{fb.key}_comment"] = fb.comment

        for run in runs:
            rid = str(run.id)
            feedback_data = feedback_by_run.get(rid, {})
            scores = {k: v for k, v in feedback_data.items() if not k.endswith("_comment")}
            outputs = run.outputs or {}
            pre_state = outputs.get("pre_computed_state") or {}

            task_id = (
                str(outputs.get("task_id") or "").strip()
                or str(pre_state.get("task_id") or "").strip()
            )

            # Extract and parse statements from feedback comments
            # Try to parse as JSON; if it fails, keep the raw comment
            statements_parsed = None
            comments_dict = {}
            
            for key, value in feedback_data.items():
                if key.endswith("_comment"):
                    comment_key = key.replace("_comment", "")
                    
                    # Try to parse as JSON
                    if value:
                        try:
                            parsed = json.loads(value)
                            comments_dict[comment_key] = parsed
                            if statements_parsed is None:
                                statements_parsed = parsed
                        except (json.JSONDecodeError, ValueError):
                            # If JSON parsing fails, keep raw comment string
                            comments_dict[comment_key] = value
                    else:
                        comments_dict[comment_key] = value

            record: Dict[str, Any] = {
                "experiment": project.name,
                "task_id": task_id,
                "thread_id": outputs.get("thread_id", ""),
                "run_id": rid,
                "scores": scores,
                "final_code": pre_state.get("final_code") or pre_state.get("generated_code") or "",
                "retrieval_context": pre_state.get("retrieval_context", ""),
                "comments": comments_dict,
                "statements": statements_parsed,
                "outputs": outputs,
            }
            # Flatten scores to top level for convenient filtering
            record.update(scores)

            records.append(record)

    print(f"Loaded {len(records)} total run(s)")
    return records

In [3]:
def filter_runs(
    records: List[Dict[str, Any]],
    **conditions: Any,
) -> List[Dict[str, Any]]:
    """
    Filter records by field values.

    Each keyword argument is a field name. The value can be:
    - A scalar     → exact equality match  (e.g., code_execution_score=0.0)
    - A callable   → predicate on the field (e.g., task_id=lambda v: "add" in v)

    Missing fields evaluate to None against the condition.

    Example:
        filter_runs(records, code_statements_score=1.0, code_execution_score=0.0)
        filter_runs(records, task_id=lambda v: v.startswith("add"))
    """
    result = []
    for record in records:
        if all(
            (cond(record.get(key)) if callable(cond) else record.get(key) == cond)
            for key, cond in conditions.items()
        ):
            result.append(record)
    return result


def display_runs(
    records: List[Dict[str, Any]],
    show_code: bool = False,
    show_context: bool = False,
    code_preview_chars: int = 600,
    context_preview_chars: int = 400,
) -> None:
    """
    Pretty-print a list of run records.

    Args:
        records:              Output of load_experiment_runs / filter_runs.
        show_code:            Print a preview of final_code.
        show_context:         Print a preview of retrieval_context.
        code_preview_chars:   Max chars to show for code preview.
        context_preview_chars: Max chars to show for context preview.
    """
    SEP = "─" * 70
    print(f"{len(records)} record(s)\n{SEP}")

    for i, r in enumerate(records, 1):
        scores = r.get("scores", {})
        score_str = "  ".join(
            f"{k}={v:.2f}" for k, v in sorted(scores.items()) if v is not None
        )
        print(f"\n[{i}]  task_id={r.get('task_id', '?')!r}")
        print(f"     experiment: {r.get('experiment', '?')}")
        print(f"     thread_id:  {r.get('thread_id', '?')}")
        print(f"     scores:     {score_str or '(none)'}")

        if show_code:
            code = r.get("final_code", "") or ""
            snippet = code[:code_preview_chars]
            ellipsis = "..." if len(code) > code_preview_chars else ""
            print(f"     final_code:\n{snippet}{ellipsis}")

        if show_context:
            ctx = r.get("retrieval_context", "") or ""
            snippet = ctx[:context_preview_chars]
            ellipsis = "..." if len(ctx) > context_preview_chars else ""
            print(f"     retrieval_context:\n{snippet}{ellipsis}")

    print(f"\n{SEP}")

## Usage Examples

In [7]:
# --- Load ---
# Empty prefix loads all experiments for the dataset.
# Supply a prefix string to scope to a specific experiment family.
EXPERIMENT_PREFIX = "COMPLEX"       # e.g. "2) With concision"
DATASET_NAME = "Dataset v2."

records = load_experiment_runs(
    experiment_prefix=EXPERIMENT_PREFIX,
    dataset_name=DATASET_NAME,
)

Found 3 experiment(s) matching prefix 'COMPLEX'
Loaded 15 total run(s)


In [8]:

# --- Verify JSON parsing is integrated into comments ---
print("=== Verifying integrated JSON parsing ===\n")

r = records[0]
print(f"Task: {r['task_id']}\n")

for comment_type, comment_value in r['comments'].items():
    print(f"[{comment_type}]")
    print(f"  Type: {type(comment_value).__name__}")
    
    if isinstance(comment_value, dict):
        print(f"  Dict keys: {list(comment_value.keys())}")
        if 'pass' in comment_value:
            print(f"  Pass: {comment_value['pass']}")
    elif isinstance(comment_value, list):
        print(f"  List length: {len(comment_value)}")
        if comment_value and isinstance(comment_value[0], dict):
            print(f"  First item keys: {list(comment_value[0].keys())}")
    else:
        print(f"  Raw string (parsing failed): {comment_value[:100]}")
    print()

# Count how many comments are parsed vs raw
parsed_count = sum(1 for r in records for c in r['comments'].values() if not isinstance(c, str))
raw_count = sum(1 for r in records for c in r['comments'].values() if isinstance(c, str))
total_comments = sum(len(r['comments']) for r in records)

print(f"Summary:")
print(f"  Total comments: {total_comments}")
print(f"  Parsed (JSON): {parsed_count}")
print(f"  Raw (string): {raw_count}")


=== Verifying integrated JSON parsing ===

Task: archive_stale_tasks

[code_execution_score]
  Type: dict
  Dict keys: ['pass', 'output', 'errors']
  Pass: True

[awrapper]
  Type: str
  Raw string (parsing failed): ValueError('Failed to extract JSON from response. Content preview: ```json\n[\n  {\n    "statement":

[rag_statements_score]
  Type: list
  List length: 6
  First item keys: ['statement', 'status', 'evidence', 'reasoning']

Summary:
  Total comments: 45
  Parsed (JSON): 44
  Raw (string): 1


In [18]:
# --- Filter: code statements correct but code didn't execute ---
interesting = filter_runs(
    records,
    #code_statements_score=lambda x: x != 1,
    code_execution_score=0.0,
)
len(interesting)

9

In [11]:
challenging_cases.keys()

dict_keys(['update_imminent_tasks', 'vague_emergency_escalation', 'propagate_status_done', 'create_task_with_blocks_and_relation'])

In [19]:
display_runs(
    interesting, 
    show_code=True, 
    show_context=False, 
    code_preview_chars=20000,
    context_preview_chars=20000
)

9 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='update_imminent_tasks'
     experiment: COMPLEX: baseline, everything applied.-0f52ddff
     thread_id:  update_imminent_tasks_20260317002230
     scores:     code_execution_score=0.00  code_statements_score=0.83  rag_statements_score=0.50
     final_code:
import os
import dotenv
import requests
from datetime import datetime, timedelta

dotenv.load_dotenv()

def bump_intensity(database_id: str = os.getenv('NOTION_TASKS_DATABASE_ID'), notion_api_key: str = os.getenv('NOTION_TOKEN')) -> bool:
    """
    Finds tasks due in exactly 3 days and updates their 'Intensity' select property to '8'.
    """
    if not database_id or not notion_api_key:
        raise ValueError("Missing database_id or notion_api_key")

    target_date = (datetime.now() + timedelta(days=3)).strftime('%Y-%m-%d')
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    headers = {
        "Authorization":

In [10]:
from collections import Counter

challenging_cases = Counter(r["task_id"] for r in interesting)
challenging_cases

Counter({'update_imminent_tasks': 3,
         'vague_emergency_escalation': 3,
         'propagate_status_done': 2,
         'create_task_with_blocks_and_relation': 1})

In [14]:
REQUIRED_KEYS = ("code_statements_score", "rag_statements_score", "code_execution_score")

counts = Counter()
total = len(records)

for r in records:
    comments = r.get("comments", {}) or {}
    for key in REQUIRED_KEYS:
        val = comments.get(key, None)
        if val is None:
            counts[f"{key}_missing"] += 1
        elif isinstance(val, str):
            counts[f"{key}_raw_or_failed_json"] += 1
        else:
            counts[f"{key}_parsed"] += 1

print(f"Total records: {total}")
for k in sorted(counts.keys()):
    print(f"{k}: {counts[k]}")

Total records: 15
code_execution_score_parsed: 15
code_statements_score_missing: 1
code_statements_score_parsed: 14
rag_statements_score_parsed: 15


In [15]:
from collections import Counter

def statement_length_counter(records, key):
    c = Counter()
    for r in records:
        comments = r.get("comments") or {}
        v = comments.get(key)
        if isinstance(v, list):
            c[len(v)] += 1
        elif v is None:
            c["missing"] += 1
        else:
            c["non_list"] += 1
    return c

for key in ("code_statements_score", "rag_statements_score"):
    cnt = statement_length_counter(records, key)
    print(f"\n{key} counts:")
    # print numeric lengths first (sorted), then other categories
    for k, v in sorted((item for item in cnt.items() if isinstance(item[0], int)), key=lambda x: x[0]):
        print(f"{k}: {v}")
    for k, v in (item for item in cnt.items() if not isinstance(item[0], int)):
        print(f"{k}: {v}")


code_statements_score counts:
6: 8
7: 3
8: 3
missing: 1

rag_statements_score counts:
6: 5
7: 1
8: 1
10: 1
12: 4
14: 1
15: 2


In [16]:
# --- Mismatch cases: code statements < 1 but execution == 1 ---
# Find cases where code has issues but still executed successfully

mismatch_cases = filter_runs(
    records,
    code_statements_score=lambda x: x is not None and x < 1,
    code_execution_score=1.0,
)

print(f"Found {len(mismatch_cases)} total mismatch cases\n")

# Group by task_id and display samples
from collections import defaultdict
by_task = defaultdict(list)
for r in mismatch_cases:
    by_task[r["task_id"]].append(r)

for task_id in sorted(by_task.keys()):
    samples = by_task[task_id]  # Get up to 2 samples
    print(f"\n{'='*70}")
    print(f"TASK: {task_id} ({len(samples)} samples shown)")
    print(f"{'='*70}")
    display_runs(
        samples,
        show_code=True,
        show_context=False,
        code_preview_chars=20000,
    )

Found 5 total mismatch cases


TASK: archive_stale_tasks (2 samples shown)
2 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='archive_stale_tasks'
     experiment: COMPLEX: baseline, everything applied.-f5eab671
     thread_id:  archive_stale_tasks_20260316004429
     scores:     code_execution_score=1.00  code_statements_score=0.50  rag_statements_score=0.75
     final_code:
import os
import dotenv
import datetime
import requests
import sys
from typing import List, Dict, Any

dotenv.load_dotenv()

def archive_stale_tasks(tasks: List[Dict[str, Any]], notion_token: str = os.getenv('NOTION_TOKEN'), notion_version: str = '2022-06-28') -> List[Dict[str, Any]]:
    """
    Identifies tasks older than 30 days, archives them, and adds a comment.
    """
    if not notion_token:
        raise ValueError("NOTION_TOKEN is not set.")

    threshold_date = datetime.date.today() - datetime.timedelta(days=30)
    headers = {
        "Authorization": f"B

In [20]:
# Filter runs with code execution == 0 and display them
failed_exec = filter_runs(records, code_execution_score=0.0)
print(f"Found {len(failed_exec)} runs with code_execution_score == 0.0\n")
display_runs(failed_exec, show_code=True, show_context=False, code_preview_chars=20000)

Found 9 runs with code_execution_score == 0.0

9 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='update_imminent_tasks'
     experiment: COMPLEX: baseline, everything applied.-0f52ddff
     thread_id:  update_imminent_tasks_20260317002230
     scores:     code_execution_score=0.00  code_statements_score=0.83  rag_statements_score=0.50
     final_code:
import os
import dotenv
import requests
from datetime import datetime, timedelta

dotenv.load_dotenv()

def bump_intensity(database_id: str = os.getenv('NOTION_TASKS_DATABASE_ID'), notion_api_key: str = os.getenv('NOTION_TOKEN')) -> bool:
    """
    Finds tasks due in exactly 3 days and updates their 'Intensity' select property to '8'.
    """
    if not database_id or not notion_api_key:
        raise ValueError("Missing database_id or notion_api_key")

    target_date = (datetime.now() + timedelta(days=3)).strftime('%Y-%m-%d')
    url = f"https://api.notion.com/v1/databases/{database_id}/